In [2]:
import akshare as ak
import inspect
import json
from datetime import datetime


def extract_akshare_apis():
    """直接从AkShare包中提取所有API函数"""

    # 存储所有API信息
    api_dict = {
        'metadata': {
            'source': 'AkShare Package',
            'version': ak.__version__ if hasattr(ak, '__version__') else 'unknown',
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'total_functions': 0
        },
        'categories': {}
    }

    # 获取akshare包中的所有属性
    all_functions = []

    for name in dir(ak):
        # 跳过私有属性和特殊属性
        if name.startswith('_'):
            continue

        try:
            obj = getattr(ak, name)

            # 只处理函数
            if callable(obj) and not inspect.isclass(obj):
                # 获取函数签名和文档
                sig = None
                doc = None

                try:
                    sig = inspect.signature(obj)
                    doc = inspect.getdoc(obj)
                except:
                    pass

                func_info = {
                    'name': name,
                    'signature': str(sig) if sig else '',
                    'docstring': doc if doc else '',
                    'parameters': []
                }

                # 解析参数
                if sig:
                    for param_name, param in sig.parameters.items():
                        param_info = {
                            'name': param_name,
                            'default': str(param.default) if param.default != inspect.Parameter.empty else None,
                            'annotation': str(param.annotation) if param.annotation != inspect.Parameter.empty else None
                        }
                        func_info['parameters'].append(param_info)

                all_functions.append(func_info)

        except Exception as e:
            print(f"处理 {name} 时出错: {e}")
            continue

    # 按类别分组函数
    categories = {
        'stock': {'name': '股票数据', 'functions': [], 'keywords': ['stock', 'shares', 'equity']},
        'fund': {'name': '基金数据', 'functions': [], 'keywords': ['fund', 'etf']},
        'bond': {'name': '债券数据', 'functions': [], 'keywords': ['bond', 'treasury']},
        'futures': {'name': '期货数据', 'functions': [], 'keywords': ['futures', 'commodity']},
        'option': {'name': '期权数据', 'functions': [], 'keywords': ['option']},
        'forex': {'name': '外汇数据', 'functions': [], 'keywords': ['forex', 'currency', 'exchange']},
        'crypto': {'name': '加密货币', 'functions': [], 'keywords': ['crypto', 'bitcoin', 'coin']},
        'macro': {'name': '宏观经济', 'functions': [], 'keywords': ['macro', 'gdp', 'cpi', 'economic']},
        'news': {'name': '新闻资讯', 'functions': [], 'keywords': ['news', 'report']},
        'index': {'name': '指数数据', 'functions': [], 'keywords': ['index']},
        'other': {'name': '其他数据', 'functions': [], 'keywords': []}
    }

    # 将函数分类
    for func in all_functions:
        func_name = func['name'].lower()
        categorized = False

        for cat_key, cat_info in categories.items():
            if cat_key == 'other':
                continue

            # 检查函数名是否包含类别关键词
            for keyword in cat_info['keywords']:
                if keyword in func_name:
                    categories[cat_key]['functions'].append(func)
                    categorized = True
                    break

            if categorized:
                break

        # 如果没有分类，放入其他类别
        if not categorized:
            categories['other']['functions'].append(func)

    # 构建最终的API字典
    for cat_key, cat_info in categories.items():
        if cat_info['functions']:
            api_dict['categories'][cat_info['name']] = {
                'count': len(cat_info['functions']),
                'functions': cat_info['functions']
            }

    api_dict['metadata']['total_functions'] = len(all_functions)

    return api_dict


def generate_html_from_apis(api_dict):
    """从API字典生成HTML文档"""

    html = """
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>AkShare API 完整文档</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }

        body {
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
            line-height: 1.6;
            color: #2c3e50;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
        }

        .container {
            max-width: 1400px;
            margin: 0 auto;
            padding: 20px;
        }

        header {
            background: white;
            border-radius: 15px;
            padding: 30px;
            margin-bottom: 30px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.1);
        }

        h1 {
            color: #667eea;
            font-size: 2.5em;
            margin-bottom: 10px;
        }

        .metadata {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 20px;
            margin-top: 20px;
            padding: 20px;
            background: #f8f9fa;
            border-radius: 10px;
        }

        .metadata-item {
            display: flex;
            flex-direction: column;
        }

        .metadata-item strong {
            color: #667eea;
            font-size: 0.9em;
            text-transform: uppercase;
            letter-spacing: 1px;
        }

        .metadata-item span {
            font-size: 1.2em;
            margin-top: 5px;
        }

        .search-container {
            background: white;
            border-radius: 15px;
            padding: 20px;
            margin-bottom: 30px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.1);
        }

        #searchBox {
            width: 100%;
            padding: 15px 20px;
            border: 2px solid #e0e0e0;
            border-radius: 10px;
            font-size: 16px;
            transition: all 0.3s;
        }

        #searchBox:focus {
            outline: none;
            border-color: #667eea;
            box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.1);
        }

        .category {
            background: white;
            border-radius: 15px;
            padding: 25px;
            margin-bottom: 25px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.1);
        }

        .category-header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 20px;
            padding-bottom: 15px;
            border-bottom: 2px solid #f0f0f0;
        }

        .category h2 {
            color: #667eea;
            font-size: 1.8em;
        }

        .function-count {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 5px 15px;
            border-radius: 20px;
            font-weight: bold;
        }

        .function {
            background: #f8f9fa;
            border-left: 4px solid #667eea;
            padding: 20px;
            margin: 15px 0;
            border-radius: 8px;
            transition: all 0.3s;
        }

        .function:hover {
            transform: translateX(5px);
            box-shadow: 0 5px 15px rgba(0,0,0,0.1);
        }

        .function-name {
            font-family: 'Courier New', monospace;
            font-size: 1.2em;
            font-weight: bold;
            color: #e74c3c;
            margin-bottom: 10px;
        }

        .function-signature {
            background: white;
            padding: 10px;
            border-radius: 5px;
            margin: 10px 0;
            font-family: monospace;
            overflow-x: auto;
        }

        .parameters {
            margin-top: 15px;
        }

        .parameter {
            display: inline-block;
            background: white;
            padding: 5px 10px;
            margin: 5px;
            border-radius: 5px;
            border: 1px solid #e0e0e0;
        }

        .parameter-name {
            color: #667eea;
            font-weight: bold;
        }

        .docstring {
            background: white;
            padding: 15px;
            border-radius: 5px;
            margin-top: 10px;
            white-space: pre-wrap;
            font-size: 0.95em;
            color: #555;
        }

        .nav-sidebar {
            position: fixed;
            right: 20px;
            top: 50%;
            transform: translateY(-50%);
            background: white;
            border-radius: 10px;
            padding: 20px;
            box-shadow: 0 5px 20px rgba(0,0,0,0.1);
            max-height: 70vh;
            overflow-y: auto;
            z-index: 100;
        }

        .nav-sidebar h3 {
            color: #667eea;
            margin-bottom: 15px;
        }

        .nav-sidebar ul {
            list-style: none;
        }

        .nav-sidebar li {
            margin: 10px 0;
        }

        .nav-sidebar a {
            color: #2c3e50;
            text-decoration: none;
            transition: color 0.3s;
        }

        .nav-sidebar a:hover {
            color: #667eea;
        }

        @media (max-width: 1200px) {
            .nav-sidebar {
                display: none;
            }
        }

        .stats-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
            gap: 20px;
            margin: 30px 0;
        }

        .stat-card {
            background: white;
            border-radius: 10px;
            padding: 20px;
            text-align: center;
            box-shadow: 0 5px 15px rgba(0,0,0,0.1);
        }

        .stat-card h3 {
            color: #667eea;
            font-size: 2em;
            margin: 10px 0;
        }

        .stat-card p {
            color: #7f8c8d;
            font-size: 0.9em;
            text-transform: uppercase;
            letter-spacing: 1px;
        }
    </style>
</head>
<body>
    <div class="container">
        <header>
            <h1>🚀 AkShare API 完整文档</h1>
            <p>自动生成的 AkShare 金融数据接口完整文档，包含所有可用函数及其参数说明。</p>

            <div class="metadata">
                <div class="metadata-item">
                    <strong>版本</strong>
                    <span>{version}</span>
                </div>
                <div class="metadata-item">
                    <strong>生成时间</strong>
                    <span>{timestamp}</span>
                </div>
                <div class="metadata-item">
                    <strong>函数总数</strong>
                    <span>{total_functions}</span>
                </div>
                <div class="metadata-item">
                    <strong>分类数量</strong>
                    <span>{category_count}</span>
                </div>
            </div>
        </header>

        <div class="search-container">
            <input type="text" id="searchBox" placeholder="🔍 搜索函数名称或描述...">
        </div>

        <div class="stats-grid">
            {stats_cards}
        </div>

        <nav class="nav-sidebar">
            <h3>快速导航</h3>
            <ul>
                {nav_items}
            </ul>
        </nav>

        <div id="content">
            {categories_html}
        </div>
    </div>

    <script>
        // 搜索功能
        document.getElementById('searchBox').addEventListener('input', function(e) {
            const searchTerm = e.target.value.toLowerCase();
            const functions = document.querySelectorAll('.function');
            const categories = document.querySelectorAll('.category');

            if (searchTerm === '') {
                functions.forEach(func => func.style.display = 'block');
                categories.forEach(cat => cat.style.display = 'block');
                return;
            }

            categories.forEach(category => {
                let hasVisibleFunction = false;
                const categoryFunctions = category.querySelectorAll('.function');

                categoryFunctions.forEach(func => {
                    const text = func.textContent.toLowerCase();
                    if (text.includes(searchTerm)) {
                        func.style.display = 'block';
                        hasVisibleFunction = true;
                    } else {
                        func.style.display = 'none';
                    }
                });

                category.style.display = hasVisibleFunction ? 'block' : 'none';
            });
        });

        // 平滑滚动
        document.querySelectorAll('a[href^="#"]').forEach(anchor => {
            anchor.addEventListener('click', function (e) {
                e.preventDefault();
                const target = document.querySelector(this.getAttribute('href'));
                if (target) {
                    target.scrollIntoView({ behavior: 'smooth', block: 'start' });
                }
            });
        });
    </script>
</body>
</html>
    """

    # 生成统计卡片
    stats_cards = []
    for cat_name, cat_data in api_dict['categories'].items():
        stats_cards.append(f"""
        <div class="stat-card">
            <h3>{cat_data['count']}</h3>
            <p>{cat_name}</p>
        </div>
        """)

    # 生成导航项
    nav_items = []
    for cat_name in api_dict['categories'].keys():
        cat_id = cat_name.replace(' ', '_')
        nav_items.append(f'<li><a href="#{cat_id}">{cat_name}</a></li>')

    # 生成分类内容
    categories_html = []
    for cat_name, cat_data in api_dict['categories'].items():
        cat_id = cat_name.replace(' ', '_')

        category_html = f"""
        <div class="category" id="{cat_id}">
            <div class="category-header">
                <h2>{cat_name}</h2>
                <span class="function-count">{cat_data['count']} 个函数</span>
            </div>
        """

        for func in cat_data['functions']:
            func_html = f"""
            <div class="function">
                <div class="function-name">ak.{func['name']}</div>
            """

            if func['signature']:
                func_html += f'<div class="function-signature">ak.{func["name"]}{func["signature"]}</div>'

            if func['parameters']:
                func_html += '<div class="parameters"><strong>参数：</strong>'
                for param in func['parameters']:
                    default_text = f" = {param['default']}" if param['default'] else ""
                    func_html += f'<span class="parameter"><span class="parameter-name">{param["name"]}</span>{default_text}</span>'
                func_html += '</div>'

            if func['docstring']:
                # 只显示前500个字符的文档
                doc_text = func['docstring'][:500] + '...' if len(func['docstring']) > 500 else func['docstring']
                func_html += f'<div class="docstring">{doc_text}</div>'

            func_html += '</div>'
            category_html += func_html

        category_html += '</div>'
        categories_html.append(category_html)

    # 替换模板占位符
    html = html.replace('{version}', str(api_dict['metadata']['version']))
    html = html.replace('{timestamp}', api_dict['metadata']['timestamp'])
    html = html.replace('{total_functions}', str(api_dict['metadata']['total_functions']))
    html = html.replace('{category_count}', str(len(api_dict['categories'])))
    html = html.replace('{stats_cards}', '\n'.join(stats_cards))
    html = html.replace('{nav_items}', '\n'.join(nav_items))
    html = html.replace('{categories_html}', '\n'.join(categories_html))

    return html


def main():
    print("📊 开始提取 AkShare API...")
    print("=" * 50)

    # 提取API
    api_dict = extract_akshare_apis()

    print(f"\n✅ 成功提取 {api_dict['metadata']['total_functions']} 个函数")
    print(f"📁 分为 {len(api_dict['categories'])} 个类别\n")

    # 显示各类别统计
    for cat_name, cat_data in api_dict['categories'].items():
        print(f"  • {cat_name}: {cat_data['count']} 个函数")

    # 保存JSON文件
    json_file = 'akshare_api_dictionary.json'
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(api_dict, f, ensure_ascii=False, indent=2)
    print(f"\n💾 JSON文件已保存: {json_file}")

    # 生成HTML文件
    html_content = generate_html_from_apis(api_dict)
    html_file = 'akshare_api_complete.html'
    with open(html_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    print(f"📄 HTML文件已保存: {html_file}")

    # 生成简化版Markdown文件
    md_file = 'akshare_api_list.md'
    with open(md_file, 'w', encoding='utf-8') as f:
        f.write("# AkShare API 函数列表\n\n")
        f.write(f"版本: {api_dict['metadata']['version']}\n")
        f.write(f"生成时间: {api_dict['metadata']['timestamp']}\n")
        f.write(f"总函数数: {api_dict['metadata']['total_functions']}\n\n")

        for cat_name, cat_data in api_dict['categories'].items():
            f.write(f"\n## {cat_name} ({cat_data['count']} 个函数)\n\n")
            for func in cat_data['functions']:
                f.write(f"- `ak.{func['name']}{func['signature']}`\n")
    print(f"📝 Markdown文件已保存: {md_file}")

    print("\n✨ 所有文件生成完成！")
    print("=" * 50)



In [3]:
try:
    import akshare as ak

    main()
except ImportError:
    print("❌ 请先安装 akshare: pip install akshare")

📊 开始提取 AkShare API...

✅ 成功提取 1046 个函数
📁 分为 11 个类别

  • 股票数据: 405 个函数
  • 基金数据: 81 个函数
  • 债券数据: 40 个函数
  • 期货数据: 63 个函数
  • 期权数据: 49 个函数
  • 外汇数据: 12 个函数
  • 加密货币: 3 个函数
  • 宏观经济: 220 个函数
  • 新闻资讯: 5 个函数
  • 指数数据: 63 个函数
  • 其他数据: 105 个函数

💾 JSON文件已保存: akshare_api_dictionary.json
📄 HTML文件已保存: akshare_api_complete.html
📝 Markdown文件已保存: akshare_api_list.md

✨ 所有文件生成完成！
